In [2]:
import pandas as pd
import numpy as np

from pathlib import Path
from scipy.spatial import cKDTree

# Project paths
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "preprocessing":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data" / "raw"
TRAIN_DIR = DATA_ROOT / "train"

# Select one trip
trip_id = "00000001"
trip_dir = TRAIN_DIR / trip_id

# Load GPS observations
track = pd.read_csv(
    trip_dir / f"{trip_id}.track",
    sep=r"\s+",
    header=None,
    names=["longitude", "latitude", "timestamp"]
)

# Load road nodes
nodes = pd.read_csv(
    trip_dir / f"{trip_id}.nodes",
    sep=r"\s+",
    header=None,
    names=["longitude", "latitude"]
)

# Load road arcs
arcs = pd.read_csv(
    trip_dir / f"{trip_id}.arcs",
    sep=r"\s+",
    header=None,
    names=["from_node_id", "to_node_id"]
)

print("GPS observations:", len(track))
print("Road nodes:", len(nodes))
print("Road arcs:", len(arcs))

GPS observations: 1315
Road nodes: 88809
Road arcs: 165790


In [3]:
# Centre of the GPS trajectory
REF_LAT = track["latitude"].mean()
REF_LON = track["longitude"].mean()

EARTH_RADIUS = 6371000

def project_coordinates(longitude, latitude):
    longitude = np.asarray(longitude)
    latitude = np.asarray(latitude)

    x = EARTH_RADIUS * np.radians(longitude - REF_LON) * np.cos(
        np.radians(REF_LAT)
    )

    y = EARTH_RADIUS * np.radians(latitude - REF_LAT)

    return np.column_stack((x, y))


# Project road nodes
node_xy = project_coordinates(
    nodes["longitude"],
    nodes["latitude"]
)

# Project GPS observations
gps_xy = project_coordinates(
    track["longitude"],
    track["latitude"]
)

# Map each arc to its start and end node coordinates
from_ids = arcs["from_node_id"].to_numpy()
to_ids = arcs["to_node_id"].to_numpy()

arc_start = node_xy[from_ids]
arc_end = node_xy[to_ids]

# Calculate arc midpoints
arc_midpoints = (arc_start + arc_end) / 2

print("GPS coordinates projected:", gps_xy.shape)
print("Road arc midpoints:", arc_midpoints.shape)

GPS coordinates projected: (1315, 2)
Road arc midpoints: (165790, 2)


In [4]:
# Spatial index over road arc midpoints
tree = cKDTree(arc_midpoints)

print("KD-tree created successfully!")

KD-tree created successfully!


In [18]:
K_NEARBY = 100
TOP_CANDIDATES = 5

# Find nearby arc midpoints for every GPS observation
_, nearby_arc_ids = tree.query(
    gps_xy,
    k=K_NEARBY
)

candidate_rows = []

for obs_id in range(len(track)):

    point = gps_xy[obs_id]

    for arc_id in nearby_arc_ids[obs_id]:

        start = arc_start[arc_id]
        end = arc_end[arc_id]

        segment = end - start

        segment_length_sq = np.dot(segment, segment)

        if segment_length_sq == 0:
            fraction = 0.0
        else:
            fraction = np.dot(point - start, segment) / segment_length_sq
            fraction = np.clip(fraction, 0, 1)

        # Closest point on the road segment
        closest_point = start + fraction * segment

        # Perpendicular GPS-to-segment distance
        distance = np.linalg.norm(point - closest_point)

        candidate_rows.append({
            "observation_id": obs_id,
            "timestamp": track.iloc[obs_id]["timestamp"],
            "gps_longitude": track.iloc[obs_id]["longitude"],
            "gps_latitude": track.iloc[obs_id]["latitude"],
            "arc_id": int(arc_id),
            "from_node_id": int(from_ids[arc_id]),
            "to_node_id": int(to_ids[arc_id]),
            "distance_m": distance,
            "projection_fraction": fraction,
            "segment_length_m": np.sqrt(segment_length_sq)
        })


candidates = pd.DataFrame(candidate_rows)

# Retain the five closest road segments for each observation
candidates = (
    candidates
    .sort_values(
        ["observation_id", "distance_m"]
    )
    .groupby("observation_id")
    .head(TOP_CANDIDATES)
    .reset_index(drop=True)
)

print("Candidate generation completed!")
print("Total GPS observations:", len(track))
print("Total candidates:", len(candidates))
print("Candidates per observation:")
print(candidates.groupby("observation_id").size().value_counts())

display(candidates.head(15))

Candidate generation completed!
Total GPS observations: 1315
Total candidates: 6575
Candidates per observation:
5    1315
Name: count, dtype: int64


,observation_id,timestamp,gps_longitude,gps_latitude,arc_id,from_node_id,to_node_id,distance_m,projection_fraction,segment_length_m
0,0,1.0,8.595203,49.844053,64973,32308,70987,0.717461,0.109780,293.233070
1,0,1.0,8.595203,49.844053,117639,61984,61983,15.011181,0.512722,1020.418564
2,0,1.0,8.595203,49.844053,2641,1584,32308,32.199208,1.000000,425.044898
3,0,1.0,8.595203,49.844053,53932,26709,24964,92.292524,0.154305,169.056399
4,0,1.0,8.595203,49.844053,50652,24964,26709,92.292524,0.845695,169.056399
5,1,2.0,8.595294,49.844441,64973,32308,70987,0.274973,0.258529,293.233070
6,1,2.0,8.595294,49.844441,117639,61984,61983,15.951541,0.469975,1020.418564
7,1,2.0,8.595294,49.844441,2641,1584,32308,75.809795,1.000000,425.044898
8,1,2.0,8.595294,49.844441,50652,24964,26709,83.407046,0.593027,169.056399
9,1,2.0,8.595294,49.844441,53932,26709,24964,83.407046,0.406973,169.056399


In [19]:
# Distance statistics for the selected candidates

print("Candidate distance statistics:")
print(candidates["distance_m"].describe().round(2))

# Best candidate distance per GPS observation
best_candidates = (
    candidates
    .groupby("observation_id")["distance_m"]
    .min()
)

print("\nBest candidate distance statistics:")
print(best_candidates.describe().round(2))

# How many GPS observations have a candidate within each threshold?
for threshold in [10, 25, 50, 100]:
    count = (best_candidates <= threshold).sum()

    print(
        f"Within {threshold} metres: "
        f"{count}/{len(track)} "
        f"({count / len(track) * 100:.2f}%)"
    )

Candidate distance statistics:
count    6575.00
mean       22.51
std        23.70
min         0.00
25%         7.15
50%        15.80
75%        28.30
max       196.23
Name: distance_m, dtype: float64

Best candidate distance statistics:
count    1315.00
mean        4.95
std         4.92
min         0.00
25%         1.88
50%         3.90
75%         6.69
max        83.24
Name: distance_m, dtype: float64
Within 10 metres: 1212/1315 (92.17%)
Within 25 metres: 1303/1315 (99.09%)
Within 50 metres: 1314/1315 (99.92%)
Within 100 metres: 1315/1315 (100.00%)


In [20]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

candidate_path = OUTPUT_DIR / f"candidates_{trip_id}.csv"

candidates.to_csv(candidate_path, index=False)

print("Candidates saved to:", candidate_path)
print("Total rows:", len(candidates))

Candidates saved to: d:\Sem 5\Machine Learning\JOB\Jackfruit\PathSnap\data\processed\candidates_00000001.csv
Total rows: 6575


In [21]:
# Find observations with the largest nearest-candidate distances

worst_observations = (
    candidates
    .groupby("observation_id")["distance_m"]
    .min()
    .sort_values(ascending=False)
    .head(15)
)

print("15 observations with the largest nearest-candidate distances:")

display(worst_observations.reset_index())

15 observations with the largest nearest-candidate distances:


,observation_id,distance_m
0,272,83.242223
1,273,44.703533
2,791,33.944294
3,473,33.475079
4,335,28.817508
5,472,28.669832
6,739,27.569441
7,255,27.477747
8,334,27.428450
9,127,26.417659


In [22]:
quality_report = pd.DataFrame({
    "metric": [
        "Total GPS observations",
        "Total candidates",
        "Candidates per observation",
        "Mean candidate distance (m)",
        "Mean nearest candidate distance (m)",
        "Observations within 10m (%)",
        "Observations within 25m (%)",
        "Observations within 50m (%)",
        "Observations within 100m (%)"
    ],
    "value": [
        len(track),
        len(candidates),
        TOP_CANDIDATES,
        candidates["distance_m"].mean(),
        best_candidates.mean(),
        (best_candidates <= 10).mean() * 100,
        (best_candidates <= 25).mean() * 100,
        (best_candidates <= 50).mean() * 100,
        (best_candidates <= 100).mean() * 100
    ]
})

quality_report.to_csv(
    OUTPUT_DIR / "candidate_quality_00000001.csv",
    index=False
)

display(quality_report)

,metric,value
0,Total GPS observations,1315.000000
1,Total candidates,6575.000000
2,Candidates per observation,5.000000
3,Mean candidate distance (m),22.511325
4,Mean nearest candidate distance (m),4.949246
5,Observations within 10m (%),92.167300
6,Observations within 25m (%),99.087452
7,Observations within 50m (%),99.923954
8,Observations within 100m (%),100.000000


In [10]:
# Inspect GPS observations with large candidate distances

outlier_ids = worst_observations.index[:10]

outlier_details = track.iloc[outlier_ids].copy()

outlier_details["best_candidate_distance_m"] = (
    best_candidates.loc[outlier_ids].values
)

display(outlier_details.sort_values("timestamp"))

,longitude,latitude,timestamp,best_candidate_distance_m
88,8.603337,49.883588,89.0,84.379608
89,8.603212,49.884109,90.0,60.721213
269,8.606502,49.961136,270.0,98.919151
270,8.606437,49.961694,271.0,94.166406
271,8.606250,49.962068,272.0,80.716796
272,8.606287,49.962462,273.0,83.242223
275,8.605862,49.963939,276.0,91.667079
276,8.605835,49.964338,277.0,68.496293
789,8.610250,50.169329,790.0,90.727691
790,8.610256,50.169620,791.0,61.832266


In [23]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "preprocessing":
    PROJECT_ROOT = PROJECT_ROOT.parent

VIS_DIR = PROJECT_ROOT / "visualisations"
VIS_DIR.mkdir(parents=True, exist_ok=True)

print("Visualisation directory:", VIS_DIR)
import folium

outlier_map = folium.Map(
    location=[REF_LAT, REF_LON],
    zoom_start=15,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri"
)

# Complete GPS trajectory
folium.PolyLine(
    list(zip(track["latitude"], track["longitude"])),
    color="blue",
    weight=3,
    tooltip="GPS trajectory"
).add_to(outlier_map)

# Highlight problematic GPS observations
for obs_id in outlier_ids:
    row = track.iloc[obs_id]

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=7,
        color="red",
        fill=True,
        tooltip=f"Observation {obs_id}"
    ).add_to(outlier_map)


    outlier_map.save(
    str(VIS_DIR / "candidate_outliers_00000001.html")
)

outlier_map

Visualisation directory: d:\Sem 5\Machine Learning\JOB\Jackfruit\PathSnap\visualisations


In [24]:
best_candidates_100 = (
    candidates
    .groupby("observation_id")["distance_m"]
    .min()
)

print("Mean nearest distance:",
      best_candidates_100.mean())

print("Median nearest distance:",
      best_candidates_100.median())

print("Observations within 10m:",
      (best_candidates_100 <= 10).sum(),
      "/", len(track))

print("Observations within 50m:",
      (best_candidates_100 <= 50).sum(),
      "/", len(track))

Mean nearest distance: 4.949245610804477
Median nearest distance: 3.902902034232211
Observations within 10m: 1212 / 1315
Observations within 50m: 1314 / 1315


In [25]:
route = pd.read_csv(
    trip_dir / f"{trip_id}.route",
    sep=r"\s+",
    header=None,
    names=["arc_id"]
)

print("Route shape:", route.shape)
display(route.head())
print("Unique route arcs:", route["arc_id"].nunique())

Route shape: (525, 1)


,arc_id
0,2641
1,64973
2,133948
3,133949
4,139208


Unique route arcs: 525


In [26]:
# Reference route arc IDs
reference_arcs = set(route["arc_id"].astype(int))

# All unique candidate arc IDs
generated_arcs = set(candidates["arc_id"].astype(int))

# Reference arcs found in generated candidates
matched_arcs = reference_arcs.intersection(generated_arcs)

coverage = len(matched_arcs) / len(reference_arcs) * 100

print("Reference route arcs:", len(reference_arcs))
print("Unique generated candidate arcs:", len(generated_arcs))
print("Reference arcs found:", len(matched_arcs))
print(f"Trip-level candidate coverage: {coverage:.2f}%")

# Identify missing reference arcs
missing_arcs = reference_arcs - generated_arcs

print("Missing reference arcs:", len(missing_arcs))

Reference route arcs: 525
Unique generated candidate arcs: 2401
Reference arcs found: 521
Trip-level candidate coverage: 99.24%
Missing reference arcs: 4


In [27]:
# Find the positions of missing arcs in the reference route
missing_route = route.reset_index(names="route_position")

missing_route = missing_route[
    missing_route["arc_id"].isin(missing_arcs)
].copy()

# Retrieve the corresponding road-network endpoints
missing_route["from_node_id"] = missing_route["arc_id"].map(
    arcs["from_node_id"]
)

missing_route["to_node_id"] = missing_route["arc_id"].map(
    arcs["to_node_id"]
)

# Calculate segment lengths in metres
missing_route["segment_length_m"] = [
    np.linalg.norm(node_xy[to_id] - node_xy[from_id])
    for from_id, to_id in zip(
        missing_route["from_node_id"],
        missing_route["to_node_id"]
    )
]

display(missing_route)

,route_position,arc_id,from_node_id,to_node_id,segment_length_m
330,330,140365,74693,74692,3.819028
331,331,140364,74692,3473,3.683788
431,431,44266,22022,22026,9.065672
475,475,97326,49641,83329,12.864847


In [28]:
display(
    track.iloc[325:337][
        ["longitude", "latitude", "timestamp"]
    ]
)

display(
    track.iloc[426:435][
        ["longitude", "latitude", "timestamp"]
    ]
)

display(
    track.iloc[470:480][
        ["longitude", "latitude", "timestamp"]
    ]
)

,longitude,latitude,timestamp
325,8.600706,49.985890,326.0
326,8.600612,49.986156,327.0
327,8.600544,49.986571,328.0
328,8.600416,49.986970,329.0
329,8.600343,49.987303,330.0
330,8.600105,49.987740,331.0
331,8.600221,49.988120,332.0
332,8.600023,49.988603,333.0
333,8.599983,49.988998,334.0
334,8.599967,49.989360,335.0


,longitude,latitude,timestamp
426,8.593521,50.030017,427.0
427,8.593705,50.030439,428.0
428,8.593733,50.030989,429.0
429,8.593912,50.031382,430.0
430,8.593999,50.031833,431.0
431,8.594332,50.032245,432.0
432,8.594472,50.032722,433.0
433,8.594677,50.033249,434.0
434,8.594892,50.033582,435.0


,longitude,latitude,timestamp
470,8.602668,50.049804,471.0
471,8.602770,50.050280,472.0
472,8.603211,50.050700,473.0
473,8.603348,50.051206,474.0
474,8.603588,50.051634,475.0
475,8.603882,50.052101,476.0
476,8.604092,50.052533,477.0
477,8.604317,50.053005,478.0
478,8.604376,50.053502,479.0
479,8.604721,50.053955,480.0


In [29]:
# Calculate movement between consecutive GPS observations
gps_dx = np.diff(gps_xy[:, 0], prepend=gps_xy[0, 0])
gps_dy = np.diff(gps_xy[:, 1], prepend=gps_xy[0, 1])

time_diff = track["timestamp"].diff().fillna(1).to_numpy()

# GPS speed in metres per second
gps_speed = np.sqrt(gps_dx**2 + gps_dy**2) / time_diff

# GPS movement direction in degrees
gps_heading = np.degrees(np.arctan2(gps_dx, gps_dy))
gps_heading = (gps_heading + 360) % 360

# Add features to the GPS observations
track["gps_speed_mps"] = gps_speed
track["gps_heading"] = gps_heading

display(track.head(10))

,longitude,latitude,timestamp,gps_speed_mps,gps_heading
0,8.595203,49.844053,1.0,0.000000,0.000000
1,8.595294,49.844441,2.0,43.629372,8.557639
2,8.595352,49.844883,3.0,49.322041,4.812529
3,8.595390,49.845379,4.0,55.219274,2.814123
4,8.595587,49.845750,5.0,43.581747,18.813500
5,8.595555,49.846167,6.0,46.424453,357.181267
6,8.595744,49.846565,7.0,46.264148,16.945032
7,8.595733,49.846931,8.0,40.704909,358.895290
8,8.595878,49.847374,9.0,50.333867,11.860123
9,8.595849,49.847684,10.0,34.532462,356.565162


In [30]:
# Calculate direction of each road segment
arc_dx = arc_end[:, 0] - arc_start[:, 0]
arc_dy = arc_end[:, 1] - arc_start[:, 1]

arc_heading = np.degrees(np.arctan2(arc_dx, arc_dy))
arc_heading = (arc_heading + 360) % 360

# Add road heading to candidate table
candidates["arc_heading"] = arc_heading[
    candidates["arc_id"].to_numpy()
]

# Attach GPS heading and speed to each candidate
candidates["gps_heading"] = candidates["observation_id"].map(
    track["gps_heading"]
)

candidates["gps_speed_mps"] = candidates["observation_id"].map(
    track["gps_speed_mps"]
)

display(candidates.head(10))

,observation_id,timestamp,gps_longitude,gps_latitude,arc_id,from_node_id,to_node_id,distance_m,projection_fraction,segment_length_m,arc_heading,gps_heading,gps_speed_mps
0,0,1.0,8.595203,49.844053,64973,32308,70987,0.717461,0.109780,293.233070,7.254223,0.000000,0.000000
1,0,1.0,8.595203,49.844053,117639,61984,61983,15.011181,0.512722,1020.418564,187.322626,0.000000,0.000000
2,0,1.0,8.595203,49.844053,2641,1584,32308,32.199208,1.000000,425.044898,7.125334,0.000000,0.000000
3,0,1.0,8.595203,49.844053,53932,26709,24964,92.292524,0.154305,169.056399,356.806678,0.000000,0.000000
4,0,1.0,8.595203,49.844053,50652,24964,26709,92.292524,0.845695,169.056399,176.806678,0.000000,0.000000
5,1,2.0,8.595294,49.844441,64973,32308,70987,0.274973,0.258529,293.233070,7.254223,8.557639,43.629372
6,1,2.0,8.595294,49.844441,117639,61984,61983,15.951541,0.469975,1020.418564,187.322626,8.557639,43.629372
7,1,2.0,8.595294,49.844441,2641,1584,32308,75.809795,1.000000,425.044898,7.125334,8.557639,43.629372
8,1,2.0,8.595294,49.844441,50652,24964,26709,83.407046,0.593027,169.056399,176.806678,8.557639,43.629372
9,1,2.0,8.595294,49.844441,53932,26709,24964,83.407046,0.406973,169.056399,356.806678,8.557639,43.629372


In [31]:
# Calculate absolute angular difference
heading_diff = (
    candidates["gps_heading"] - candidates["arc_heading"]
)

candidates["heading_difference"] = (
    (heading_diff + 180) % 360
) - 180

candidates["heading_difference"] = (
    candidates["heading_difference"].abs()
)

display(
    candidates[
        [
            "observation_id",
            "arc_id",
            "distance_m",
            "gps_heading",
            "arc_heading",
            "heading_difference"
        ]
    ].head(15)
)

,observation_id,arc_id,distance_m,gps_heading,arc_heading,heading_difference
0,0,64973,0.717461,0.000000,7.254223,7.254223
1,0,117639,15.011181,0.000000,187.322626,172.677374
2,0,2641,32.199208,0.000000,7.125334,7.125334
3,0,53932,92.292524,0.000000,356.806678,3.193322
4,0,50652,92.292524,0.000000,176.806678,176.806678
5,1,64973,0.274973,8.557639,7.254223,1.303415
6,1,117639,15.951541,8.557639,187.322626,178.764987
7,1,2641,75.809795,8.557639,7.125334,1.432304
8,1,50652,83.407046,8.557639,176.806678,168.249039
9,1,53932,83.407046,8.557639,356.806678,11.750961


In [32]:
# Mark movement features unavailable for the first observation
candidates.loc[
    candidates["observation_id"] == 0,
    ["gps_speed_mps", "gps_heading", "heading_difference"]
] = np.nan

In [33]:
feature_columns = [
    "observation_id",
    "timestamp",
    "arc_id",
    "from_node_id",
    "to_node_id",
    "distance_m",
    "projection_fraction",
    "segment_length_m",
    "gps_speed_mps",
    "gps_heading",
    "arc_heading",
    "heading_difference"
]

candidate_features = candidates[feature_columns].copy()

print("Feature table shape:", candidate_features.shape)
print("\nMissing values:")
print(candidate_features.isnull().sum())

display(candidate_features.head())

Feature table shape: (6575, 12)

Missing values:
observation_id         0
timestamp              0
arc_id                 0
from_node_id           0
to_node_id             0
distance_m             0
projection_fraction    0
segment_length_m       0
gps_speed_mps          5
gps_heading            5
arc_heading            0
heading_difference     5
dtype: int64


,observation_id,timestamp,arc_id,from_node_id,to_node_id,distance_m,projection_fraction,segment_length_m,gps_speed_mps,gps_heading,arc_heading,heading_difference
0,0,1.0,64973,32308,70987,0.717461,0.109780,293.233070,NaN,NaN,7.254223,NaN
1,0,1.0,117639,61984,61983,15.011181,0.512722,1020.418564,NaN,NaN,187.322626,NaN
2,0,1.0,2641,1584,32308,32.199208,1.000000,425.044898,NaN,NaN,7.125334,NaN
3,0,1.0,53932,26709,24964,92.292524,0.154305,169.056399,NaN,NaN,356.806678,NaN
4,0,1.0,50652,24964,26709,92.292524,0.845695,169.056399,NaN,NaN,176.806678,NaN


In [34]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DIR / "candidate_features_00000001.csv"

candidate_features.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(candidate_features))
print("Features:", len(candidate_features.columns))

Saved: d:\Sem 5\Machine Learning\JOB\Jackfruit\PathSnap\data\processed\candidate_features_00000001.csv
Rows: 6575
Features: 12


In [35]:
def process_trip(trip_id):
    trip_dir = TRAIN_DIR / trip_id

    # Load data
    track = pd.read_csv(
        trip_dir / f"{trip_id}.track",
        sep=r"\s+", header=None,
        names=["longitude", "latitude", "timestamp"]
    )

    nodes = pd.read_csv(
        trip_dir / f"{trip_id}.nodes",
        sep=r"\s+", header=None,
        names=["longitude", "latitude"]
    )

    arcs = pd.read_csv(
        trip_dir / f"{trip_id}.arcs",
        sep=r"\s+", header=None,
        names=["from_node_id", "to_node_id"]
    )

    # Project coordinates
    ref_lat = track["latitude"].mean()
    ref_lon = track["longitude"].mean()

    def project(lon, lat):
        x = EARTH_RADIUS * np.radians(
            np.asarray(lon) - ref_lon
        ) * np.cos(np.radians(ref_lat))

        y = EARTH_RADIUS * np.radians(
            np.asarray(lat) - ref_lat
        )

        return np.column_stack((x, y))

    node_xy = project(nodes["longitude"], nodes["latitude"])
    gps_xy = project(track["longitude"], track["latitude"])

    from_ids = arcs["from_node_id"].to_numpy()
    to_ids = arcs["to_node_id"].to_numpy()

    arc_start = node_xy[from_ids]
    arc_end = node_xy[to_ids]

    arc_midpoints = (arc_start + arc_end) / 2

    # Build KD-tree
    tree = cKDTree(arc_midpoints)

    _, nearby_arc_ids = tree.query(gps_xy, k=100)

    candidate_rows = []

    # Generate candidates
    for obs_id in range(len(track)):
        point = gps_xy[obs_id]

        for arc_id in nearby_arc_ids[obs_id]:
            start = arc_start[arc_id]
            end = arc_end[arc_id]

            segment = end - start
            length_sq = np.dot(segment, segment)

            if length_sq == 0:
                fraction = 0.0
            else:
                fraction = np.dot(
                    point - start, segment
                ) / length_sq

                fraction = np.clip(fraction, 0, 1)

            closest = start + fraction * segment
            distance = np.linalg.norm(point - closest)

            candidate_rows.append({
                "trip_id": trip_id,
                "observation_id": obs_id,
                "timestamp": track.iloc[obs_id]["timestamp"],
                "arc_id": int(arc_id),
                "from_node_id": int(from_ids[arc_id]),
                "to_node_id": int(to_ids[arc_id]),
                "distance_m": distance,
                "projection_fraction": fraction,
                "segment_length_m": np.sqrt(length_sq)
            })

    candidates = pd.DataFrame(candidate_rows)

    candidates = (
        candidates.sort_values(
            ["observation_id", "distance_m"]
        )
        .groupby("observation_id")
        .head(5)
        .reset_index(drop=True)
    )

    # GPS movement features
    gps_dx = np.diff(gps_xy[:, 0], prepend=gps_xy[0, 0])
    gps_dy = np.diff(gps_xy[:, 1], prepend=gps_xy[0, 1])

    time_diff = track["timestamp"].diff().fillna(1).to_numpy()

    gps_speed = np.sqrt(gps_dx**2 + gps_dy**2) / time_diff

    gps_heading = (
        np.degrees(np.arctan2(gps_dx, gps_dy)) + 360
    ) % 360

    track["gps_speed_mps"] = gps_speed
    track["gps_heading"] = gps_heading

    # Road heading
    arc_dx = arc_end[:, 0] - arc_start[:, 0]
    arc_dy = arc_end[:, 1] - arc_start[:, 1]

    arc_heading = (
        np.degrees(np.arctan2(arc_dx, arc_dy)) + 360
    ) % 360

    candidates["arc_heading"] = arc_heading[
        candidates["arc_id"].to_numpy()
    ]

    candidates["gps_heading"] = candidates["observation_id"].map(
        track["gps_heading"]
    )

    candidates["gps_speed_mps"] = candidates["observation_id"].map(
        track["gps_speed_mps"]
    )

    # Heading difference
    difference = (
        candidates["gps_heading"] - candidates["arc_heading"]
    )

    candidates["heading_difference"] = (
        ((difference + 180) % 360) - 180
    ).abs()

    # First GPS observation has no previous movement
    candidates.loc[
        candidates["observation_id"] == 0,
        ["gps_speed_mps", "gps_heading", "heading_difference"]
    ] = np.nan

    return candidates

In [36]:
trip_ids = sorted(
    folder.name
    for folder in TRAIN_DIR.iterdir()
    if folder.is_dir()
)

print("Training trips found:", len(trip_ids))

Training trips found: 80


In [38]:
all_candidates = []

for i, trip_id in enumerate(trip_ids, start=1):
    try:
        trip_candidates = process_trip(trip_id)
        all_candidates.append(trip_candidates)

        print(
            f"[{i}/{len(trip_ids)}] "
            f"Processed {trip_id} - "
            f"{len(trip_candidates)} candidates"
        )

    except Exception as e:
        print(f"Error processing {trip_id}: {e}")

combined_features = pd.concat(
    all_candidates,
    ignore_index=True
)

print("\nProcessing complete!")
print("Trips processed:", len(all_candidates))
print("Total candidate rows:", len(combined_features))
print("Total features:", len(combined_features.columns))

[1/80] Processed 00000001 - 6575 candidates
[2/80] Processed 00000002 - 36215 candidates
[3/80] Processed 00000003 - 4105 candidates
[4/80] Processed 00000004 - 5320 candidates
[5/80] Processed 00000005 - 5225 candidates
[6/80] Processed 00000006 - 7250 candidates
[7/80] Processed 00000008 - 15465 candidates
[8/80] Processed 00000010 - 13555 candidates
[9/80] Processed 00000011 - 14140 candidates
[10/80] Processed 00000012 - 8040 candidates
[11/80] Processed 00000013 - 12900 candidates
[12/80] Processed 00000014 - 2240 candidates
[13/80] Processed 00000016 - 6525 candidates
[14/80] Processed 00000018 - 1895 candidates
[15/80] Processed 00000019 - 11770 candidates
[16/80] Processed 00000020 - 11765 candidates
[17/80] Processed 00000021 - 78915 candidates
[18/80] Processed 00000022 - 2370 candidates
[19/80] Processed 00000024 - 4005 candidates
[20/80] Processed 00000025 - 5895 candidates
[21/80] Processed 00000026 - 1805 candidates
[22/80] Processed 00000027 - 3470 candidates
[23/80] Pro

In [39]:
# Inspect variables created during candidate generation

print("Available candidate-related variables:")

for name, value in globals().copy().items():
    if "candidate" in name.lower():
        if isinstance(value, pd.DataFrame):
            print(f"{name}: DataFrame, shape = {value.shape}")
        elif isinstance(value, list):
            print(f"{name}: List, length = {len(value)}")

Available candidate-related variables:
candidate_rows: List, length = 131500
candidates: DataFrame, shape = (6575, 14)
candidate_features: DataFrame, shape = (6575, 12)
all_candidates: List, length = 80
trip_candidates: DataFrame, shape = (11610, 13)


In [40]:
# Combine all 80 processed trips
combined_features = pd.concat(
    all_candidates,
    ignore_index=True
)

print("Combined dataset shape:", combined_features.shape)
print("Unique trips:", combined_features["trip_id"].nunique())

print("\nMissing values:")
print(combined_features.isnull().sum())

print("\nDuplicate rows:",
      combined_features.duplicated().sum())

print("\nCandidates per observation:")
print(
    combined_features.groupby(
        ["trip_id", "observation_id"]
    ).size().value_counts().sort_index()
)

Combined dataset shape: (1042085, 13)
Unique trips: 80

Missing values:
trip_id                  0
observation_id           0
timestamp                0
arc_id                   0
from_node_id             0
to_node_id               0
distance_m               0
projection_fraction      0
segment_length_m         0
arc_heading              0
gps_heading            400
gps_speed_mps          400
heading_difference     400
dtype: int64

Duplicate rows: 0

Candidates per observation:
5    208417
Name: count, dtype: int64


In [41]:
# Overall candidate distance statistics
print("OVERALL DISTANCE STATISTICS")
display(
    combined_features["distance_m"].describe()
)

# Best candidate distance for each GPS observation
best_per_observation = (
    combined_features
    .groupby(["trip_id", "observation_id"])["distance_m"]
    .min()
)

print("\nBEST CANDIDATE DISTANCE")
display(best_per_observation.describe())

# Distance thresholds
print("\nCANDIDATE QUALITY")

for threshold in [10, 25, 50, 100]:
    count = (best_per_observation <= threshold).sum()
    percentage = count / len(best_per_observation) * 100

    print(
        f"Within {threshold}m: "
        f"{count}/{len(best_per_observation)} "
        f"({percentage:.2f}%)"
    )

OVERALL DISTANCE STATISTICS


count    1.042085e+06
mean     4.213870e+01
std      7.651492e+01
min      0.000000e+00
25%      6.223202e+00
50%      1.558189e+01
75%      4.705282e+01
max      1.448808e+03
Name: distance_m, dtype: float64


BEST CANDIDATE DISTANCE


count    208417.000000
mean         12.831483
std          23.687871
min           0.000000
25%           2.292657
50%           4.989835
75%           9.361650
max         256.860281
Name: distance_m, dtype: float64


CANDIDATE QUALITY
Within 10m: 160846/208417 (77.18%)
Within 25m: 186800/208417 (89.63%)
Within 50m: 190070/208417 (91.20%)
Within 100m: 207003/208417 (99.32%)


In [42]:
# Find the 20 observations with the largest nearest-candidate distances

worst_observations = (
    best_per_observation
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

worst_observations.columns = [
    "trip_id",
    "observation_id",
    "best_candidate_distance_m"
]

display(worst_observations)

,trip_id,observation_id,best_candidate_distance_m
0,00000096,1620,256.860281
1,00000096,1615,253.079653
2,00000096,1623,252.190094
3,00000096,1619,252.190094
4,00000096,1617,251.078144
5,00000096,1618,250.410975
6,00000096,1622,247.297517
7,00000096,1621,247.297517
8,00000096,1616,246.852737
9,00000096,1614,246.185568


In [43]:
# Load trip 00000096 independently
debug_trip_id = "00000096"
debug_dir = TRAIN_DIR / debug_trip_id

debug_track = pd.read_csv(
    debug_dir / f"{debug_trip_id}.track",
    sep=r"\s+",
    header=None,
    names=["longitude", "latitude", "timestamp"]
)

# Inspect the problematic GPS observations
display(
    debug_track.iloc[1609:1630]
)

,longitude,latitude,timestamp
1609,-95.662748,36.164533,1610.0
1610,-95.662483,36.164568,1611.0
1611,-95.662183,36.164539,1612.0
1612,-95.661803,36.164489,1613.0
1613,-95.661547,36.164571,1614.0
1614,-95.661164,36.164512,1615.0
1615,-95.660815,36.164574,1616.0
1616,-95.660434,36.164518,1617.0
1617,-95.660097,36.164556,1618.0
1618,-95.659791,36.164550,1619.0


In [44]:
debug_nodes = pd.read_csv(
    debug_dir / f"{debug_trip_id}.nodes",
    sep=r"\s+",
    header=None,
    names=["longitude", "latitude"]
)

print("GPS coordinate ranges:")
print(debug_track[["longitude", "latitude"]].agg(["min", "max"]))

print("\nRoad node coordinate ranges:")
print(debug_nodes[["longitude", "latitude"]].agg(["min", "max"]))

GPS coordinate ranges:
     longitude   latitude
min -95.869374  36.137525
max -95.331981  36.262757

Road node coordinate ranges:
     longitude   latitude
min -95.879309  36.127533
max -95.322158  36.272715


In [45]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

final_path = PROCESSED_DIR / "all_training_candidate_features.csv"

combined_features.to_csv(final_path, index=False)

print("Saved:", final_path)
print("Rows:", len(combined_features))
print("Columns:", len(combined_features.columns))
print("Trips:", combined_features["trip_id"].nunique())

Saved: d:\Sem 5\Machine Learning\JOB\Jackfruit\PathSnap\data\processed\all_training_candidate_features.csv
Rows: 1042085
Columns: 13
Trips: 80


In [46]:
coverage_results = []

for trip_id in trip_ids:
    trip_dir = TRAIN_DIR / trip_id

    route = pd.read_csv(
        trip_dir / f"{trip_id}.route",
        sep=r"\s+",
        header=None,
        names=["arc_id"]
    )

    reference_arcs = set(route["arc_id"].astype(int))

    generated_arcs = set(
        combined_features.loc[
            combined_features["trip_id"] == trip_id,
            "arc_id"
        ].astype(int)
    )

    matched = reference_arcs.intersection(generated_arcs)

    coverage_results.append({
        "trip_id": trip_id,
        "reference_arcs": len(reference_arcs),
        "matched_arcs": len(matched),
        "coverage_pct": (
            len(matched) / len(reference_arcs) * 100
        )
    })

coverage_df = pd.DataFrame(coverage_results)

display(coverage_df)

print(
    "\nMean trip-level coverage:",
    round(coverage_df["coverage_pct"].mean(), 2),
    "%"
)

print(
    "Overall trip-level coverage:",
    round(
        coverage_df["matched_arcs"].sum()
        / coverage_df["reference_arcs"].sum() * 100,
        2
    ),
    "%"
)

,trip_id,reference_arcs,matched_arcs,coverage_pct
0,00000001,525,521,99.238095
1,00000002,262,262,100.000000
2,00000003,79,79,100.000000
3,00000004,320,301,94.062500
4,00000005,105,105,100.000000
...,...,...,...,...
75,00000095,299,296,98.996656
76,00000096,456,455,99.780702
77,00000097,444,441,99.324324
78,00000098,1003,999,99.601196



Mean trip-level coverage: 98.43 %
Overall trip-level coverage: 98.23 %
